# Ejercicios UD03_02

## Clasificar preguntas

En la práctica [Clasificación de texto con PyTorch](https://colab.research.google.com/github/martinezpenya/MIA-IABD-2425/blob/main/UD03/notebooks/2.-classificacio_text_torch_ES.ipynb) hemos visto el proceso para convertir un texto en una representación numérica que pueda ser utilizada por un algoritmo de aprendizaje automático. Hemos visto diferentes representaciones como *Bolsa de palabras* (BoW) y *incrustaciones de palabras* (word embeddings) y cómo entrenar una red neuronal para clasificar texto.

En esta práctica, deberá repetir el proceso para clasificar las preguntas en temas. Usaremos el conjunto de datos `Trec` que contiene preguntas en inglés y su tema. El conjunto de datos está disponible en [trec](https://huggingface.co/datasets/CogComp/trec).

> Para evitar problemas con la descarga del dataset Trec, debereis hacer un downgrade de la libreria datasets: `datasets==3.6.0`

### Objetivos de la práctica
* Reproducir el proceso visto en la práctica [Clasificación de texto con PyTorch](https://colab.research.google.com/github/martinezpenya/MIA-IABD-2425/blob/main/UD03/notebooks/2.-classificacio_text_torch_ES.ipynb) para clasificar preguntas en temáticas.
* Deberá preparar una red neuronal con PyTorch para clasificar las preguntas.
* Pruebe las diferentes representaciones vistas para convertir el texto en una representación numérica.
* Tendrá que comparar los resultados obtenidos con las diferentes representaciones.

In [3]:
pip install datasets==3.6.0

In [4]:
from datasets import load_dataset

dataset = load_dataset("trec")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

The repository for trec contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/trec.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/5452 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

{'text': 'How did serfdom develop in and then leave Russia ?', 'coarse_label': 2, 'fine_label': 26}


In [5]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

train_texts = [preprocess(x["text"]) for x in train_data]
test_texts = [preprocess(x["text"]) for x in test_data]

train_labels = [x["coarse_label"] for x in train_data]
test_labels = [x["coarse_label"] for x in test_data]

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
X_train_bow = vectorizer.fit_transform(train_texts).toarray()
X_test_bow = vectorizer.transform(test_texts).toarray()

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class TrecDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TrecDataset(X_train_bow, train_labels)
test_dataset = TrecDataset(X_test_bow, test_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [8]:
import torch.nn as nn

class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model_bow = BoWClassifier(5000, 6)

In [9]:
def train_model(model, train_loader, test_loader, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                preds = model(X).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        acc = correct / total
        print(f"Epoch {epoch+1} | Accuracy: {acc:.4f}")

train_model(model_bow, train_loader, test_loader)

Epoch 1 | Accuracy: 0.7500
Epoch 2 | Accuracy: 0.8320
Epoch 3 | Accuracy: 0.8520
Epoch 4 | Accuracy: 0.8660
Epoch 5 | Accuracy: 0.8780


In [10]:
from collections import Counter

def tokenize(text):
    return text.split()

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

vocab = {"<pad>": 0}
for word in counter:
    vocab[word] = len(vocab)

def encode(text, vocab, max_len=20):
    tokens = tokenize(text)
    ids = [vocab.get(t, 0) for t in tokens][:max_len]
    return ids + [0] * (max_len - len(ids))

X_train_emb = [encode(t, vocab) for t in train_texts]
X_test_emb = [encode(t, vocab) for t in test_texts]

In [11]:
class TrecEmbDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset_emb = TrecEmbDataset(X_train_emb, train_labels)
test_dataset_emb = TrecEmbDataset(X_test_emb, test_labels)

train_loader_emb = DataLoader(train_dataset_emb, batch_size=32, shuffle=True)
test_loader_emb = DataLoader(test_dataset_emb, batch_size=32)

In [12]:
class EmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        pooled = embedded.mean(dim=1)
        return self.fc(pooled)

model_emb = EmbeddingClassifier(len(vocab), 100, 6)

In [13]:
train_model(model_emb, train_loader_emb, test_loader_emb)

Epoch 1 | Accuracy: 0.3500
Epoch 2 | Accuracy: 0.5060
Epoch 3 | Accuracy: 0.5740
Epoch 4 | Accuracy: 0.5780
Epoch 5 | Accuracy: 0.6340
